# Base 5 — EDA, limpeza e preparação: preços diários do ouro

Este notebook faz a ingestão, as verificações de qualidade, o EDA inicial e a preparação de uma tabela reutilizável para modelos de séries temporais.

**Avaliação:** o corte é cronológico, com 80% iniciais para treino e 20% finais para teste. Não há embaralhamento. `RANDOM_STATE = 42` é reservado para algoritmos e procedimentos estocásticos; ele não define o corte temporal.

In [ ]:
from pathlib import Path
import hashlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
TRAIN_RATIO, TEST_RATIO = 0.80, 0.20
assert TRAIN_RATIO + TEST_RATIO == 1.0

DATA_URL = ('https://raw.githubusercontent.com/dengyishuo/'
            'quantitative-finance/master/gold.daily.prices.csv')
LOCAL_DATA_PATH = Path('gold.daily.prices.csv')
pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-whitegrid')
print(f'RANDOM_STATE={RANDOM_STATE} | divisão temporal: {TRAIN_RATIO:.0%}/{TEST_RATIO:.0%}')

## 1. Ingestão

O arquivo possui extensão CSV, mas seus campos são separados por espaços em branco. O símbolo `.` representa ausência de cotação. Para reprodutibilidade máxima, salve uma cópia local do arquivo e registre o hash; se a cópia local não existir, a célula usa a fonte remota.

In [ ]:
source = LOCAL_DATA_PATH if LOCAL_DATA_PATH.exists() else DATA_URL
df_raw = pd.read_csv(source, sep=r'\s+', na_values='.', dtype={'DATE': 'string'})

file_hash = (hashlib.sha256(LOCAL_DATA_PATH.read_bytes()).hexdigest()
             if LOCAL_DATA_PATH.exists()
             else 'não calculado: fonte remota')
print(f'Fonte: {source}')
print(f'Linhas: {len(df_raw):,} | colunas: {df_raw.columns.tolist()}')
print(f'SHA-256: {file_hash}')
display(df_raw.head())

## 2. Limpeza e relatório de qualidade

Não há interpolação de preços. Fins de semana, feriados e dias sem cotação não devem ser confundidos com valor zero. As linhas sem preço são preservadas na auditoria e retiradas apenas da tabela modelável.

In [ ]:
df = df_raw.copy()
df['DATE'] = pd.to_datetime(df['DATE'], format='%Y-%m-%d', errors='coerce')
df['VALUE'] = pd.to_numeric(df['VALUE'], errors='coerce')
df = df.sort_values('DATE', kind='stable').reset_index(drop=True)

quality_report = pd.DataFrame({'métrica': [
    'linhas', 'datas_nulas', 'valores_nulos', 'datas_duplicadas',
    'valores_não_positivos', 'primeira_data', 'última_data'],
    'valor': [len(df), int(df.DATE.isna().sum()), int(df.VALUE.isna().sum()),
              int(df.DATE.duplicated().sum()), int((df.VALUE.dropna() <= 0).sum()),
              df.DATE.min(), df.DATE.max()]})
display(quality_report)

assert df.DATE.notna().all(), 'Há datas inválidas; investigar antes de continuar.'
assert not df.DATE.duplicated().any(), 'Há datas duplicadas; definir consolidação.'
missing_price_dates = df.loc[df.VALUE.isna(), ['DATE']].rename(columns={'DATE': 'data_sem_cotação'})
display(missing_price_dates.head(10))

df_clean = (df.dropna(subset=['VALUE']).loc[lambda x: x.VALUE > 0]
              .copy().reset_index(drop=True))
df_clean['gap_days'] = df_clean.DATE.diff().dt.days
print(f'Linhas utilizáveis: {len(df_clean):,}')
print(f'Maior intervalo entre cotações: {int(df_clean.gap_days.max())} dias')

## 3. EDA inicial

A coluna `VALUE` não declara, no próprio arquivo, unidade, moeda ou fornecedor original. Portanto, este notebook a apresenta como valor de cotação na unidade de origem; a unidade precisa ser confirmada antes de interpretações financeiras.

In [ ]:
eda = df_clean[['DATE', 'VALUE']].copy()
eda['log_return_1d'] = np.log(eda.VALUE).diff()

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
axes[0, 0].plot(eda.DATE, eda.VALUE, lw=.8, color='#b58900')
axes[0, 0].set(title='Histórico da cotação', xlabel='Data', ylabel='VALUE (unidade de origem)')
axes[0, 1].plot(eda.DATE, eda.log_return_1d, lw=.5, color='#268bd2')
axes[0, 1].axhline(0, color='black', lw=.7)
axes[0, 1].set(title='Retorno logarítmico por cotação', xlabel='Data', ylabel='log-retorno')
axes[1, 0].hist(eda.log_return_1d.dropna(), bins=80, color='#268bd2', edgecolor='white')
axes[1, 0].set(title='Distribuição dos retornos', xlabel='log-retorno', ylabel='Frequência')
axes[1, 1].plot(eda.DATE, eda.VALUE.rolling(252, min_periods=30).std(), lw=.8, color='#859900')
axes[1, 1].set(title='Volatilidade do preço em janela de 252 cotações', xlabel='Data', ylabel='desvio-padrão')
plt.tight_layout(); plt.show()

annual_summary = eda.set_index('DATE').VALUE.resample('YE').agg(['count', 'mean', 'min', 'max', 'last'])
annual_summary.index = annual_summary.index.year
annual_summary.index.name = 'ano'
display(annual_summary.tail(10))
display(eda.log_return_1d.describe(percentiles=[.01, .05, .5, .95, .99]).to_frame('log_return_1d'))

## 4. Tabela preparada para múltiplos modelos

Cada linha representa o instante `t`. As features usam dados observados até `t`; os dois alvos são o próximo registro disponível, `t+1`. Janelas móveis são defasadas em uma observação para explicitar que não usam informação futura. Se for escolhido um alvo, use somente aquele alvo por experimento.

In [ ]:
prepared = df_clean[['DATE', 'VALUE', 'gap_days']].rename(columns={'VALUE': 'price_t'}).copy()
prepared['log_price_t'] = np.log(prepared.price_t)
prepared['log_return_t'] = prepared.log_price_t.diff()

for lag in (1, 2, 5, 10, 21):
    prepared[f'price_lag_{lag}'] = prepared.price_t.shift(lag)
    prepared[f'return_lag_{lag}'] = prepared.log_return_t.shift(lag)
for window in (5, 10, 21):
    prepared[f'price_ma_{window}'] = prepared.price_t.shift(1).rolling(window).mean()
    prepared[f'return_vol_{window}'] = prepared.log_return_t.shift(1).rolling(window).std()

prepared['day_of_week'] = prepared.DATE.dt.dayofweek
prepared['month'] = prepared.DATE.dt.month
prepared['month_sin'] = np.sin(2 * np.pi * prepared.month / 12)
prepared['month_cos'] = np.cos(2 * np.pi * prepared.month / 12)
prepared['target_date'] = prepared.DATE.shift(-1)
prepared['target_price_t_plus_1'] = prepared.price_t.shift(-1)
prepared['target_log_return_t_plus_1'] = np.log(prepared.target_price_t_plus_1) - prepared.log_price_t

target_columns = ['target_price_t_plus_1', 'target_log_return_t_plus_1']
feature_columns = [c for c in prepared.columns if c not in ['DATE', 'target_date', *target_columns]]
model_df = prepared.dropna(subset=feature_columns + target_columns).reset_index(drop=True)
print(f'Linhas modeláveis: {len(model_df):,}')
print(f'Features ({len(feature_columns)}): {feature_columns}')
display(model_df.head())

## 5. Separação temporal 80% / 20%

O teste é sempre o trecho final da série. Não utilizar `train_test_split` com o comportamento padrão, pois o embaralhamento geraria uma avaliação irrealista.

In [ ]:
split_index = int(len(model_df) * TRAIN_RATIO)
train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()
split_metadata = pd.DataFrame({'conjunto': ['treino', 'teste'],
    'linhas': [len(train_df), len(test_df)],
    'proporção': [len(train_df) / len(model_df), len(test_df) / len(model_df)],
    'início': [train_df.DATE.min(), test_df.DATE.min()],
    'fim': [train_df.DATE.max(), test_df.DATE.max()]})
display(split_metadata)
assert train_df.DATE.max() < test_df.DATE.min()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_df.DATE, train_df.price_t, label='Treino', lw=.8, color='#2aa198')
ax.plot(test_df.DATE, test_df.price_t, label='Teste', lw=.8, color='#dc322f')
ax.axvline(test_df.DATE.min(), color='black', ls='--', lw=1, label='Corte 80/20')
ax.set(title='Corte cronológico para modelagem', xlabel='Data', ylabel='VALUE (unidade de origem)')
ax.legend(); plt.show()

## 6. Objetos finais e próximos passos

As matrizes são mantidas sem escala porque serão utilizadas por modelos diferentes. Qualquer imputador, escalador e seleção de variáveis deve ser ajustado apenas em `X_train` — idealmente dentro de um `Pipeline` e da validação temporal expansiva. Mantenha `test_df` intocado até a avaliação final.

In [ ]:
# Alvo-padrão: retorno do próximo pregão. Para preço, use 'target_price_t_plus_1'.
TARGET = 'target_log_return_t_plus_1'
X_train, y_train = train_df[feature_columns].copy(), train_df[TARGET].copy()
X_test, y_test = test_df[feature_columns].copy(), test_df[TARGET].copy()

# Baselines a usar na fase de modelagem.
baseline_return_test = pd.Series(0.0, index=y_test.index, name='return_zero')
baseline_price_test = test_df.price_t.rename('persistence_price')

print(f'Alvo: {TARGET}')
print(f'X_train: {X_train.shape}; y_train: {y_train.shape}')
print(f'X_test: {X_test.shape}; y_test: {y_test.shape}')
display(X_train.head())

# Próximo passo: validar modelos em walk-forward dentro de train_df;
# em modelos estocásticos, utilizar random_state=RANDOM_STATE.